In [53]:
# Imports & NLP Code
import pandas as pd
from transformers import pipeline
import pandas as pd
import requests
from tqdm import tqdm
import inflect  

def process_text(text):    
    tdf1 = _tag_genes(text)
    tdf2 = _tag_chemicals(text)
    tdf3 = _tag_diseases(text)
    df = pd.concat([tdf1,tdf2,tdf3])
    df = _drop_unknowns(df)
    df = normalize(df)
    return df

def batch(corpus):
    df = pd.DataFrame()
    for entry in corpus:
        tdf1 = _tag_genes(entry)
        tdf2 = _tag_chemicals(entry)
        tdf3 = _tag_diseases(entry)
        df = pd.concat([df,tdf1,tdf2,tdf3])
    df = _drop_unknowns(df)
    df['start'] = df['start'].astype(int)
    df['end'] = df['end'].astype(int)
    df = normalize(df)
    return df

def normalize(result):
    result['concept_match_type'] = None
    result['concept_id'] = None
    result['concept_label'] = None
    for index, row in tqdm(result.iterrows()):
        word = _singularize(row['word'])
        if row['entity_group'] == 'GENETIC':
            norm_result = _normalize_gene(word)
        if row['entity_group'] == 'CHEMICAL':
            norm_result = _normalize_therapy(word)
        if row['entity_group'] == 'DISEASE':
            norm_result = _normalize_disease(word)
        result.at[index, 'concept_match_type'] = norm_result[0]
        result.at[index, 'concept_id'] = norm_result[1]
        result.at[index, 'concept_label'] = norm_result[2]
    return result


def _singularize(word):
    inflector = inflect.engine()
    return inflector.singular_noun(word) or word

# TODO: Implement cached queries to improve normalization time -- Brian Walsh/ Kori / James
def _normalize_gene(word):
    r = requests.get(f'https://normalize.cancervariants.org/gene/normalize?q={word}')
    response = r.json()
    if response['match_type'] != 0:
        match_type = response['match_type']
        concept_id = response['gene']['id']
        label = response['gene']['name']
    else:
        match_type = response['match_type']
        concept_id = None
        label = None

    return [match_type, concept_id, label]

def _normalize_disease(word):
    r = requests.get(f'https://normalize.cancervariants.org/disease/normalize?q={word}')
    response = r.json()
    if response['match_type'] != 0:
        match_type = response['match_type']
        concept_id = response['disease']['id']
        label = response['disease']['name']
    else:
        match_type = response['match_type']
        concept_id = None
        label = None
    
    return [match_type, concept_id, label]

def _normalize_therapy(word):
    r = requests.get(f'https://normalize.cancervariants.org/therapy/normalize?q={word}&infer_namespace=true')
    response = r.json()
    if response['match_type'] != 0:
        match_type = response['match_type']
        concept_id = response['therapy']['id']
        label = response['therapy']['name']
    else:
        match_type = response['match_type']
        concept_id = None
        label = None

    return [match_type, concept_id, label]


def _drop_unknowns(result):
    try:
        dropped = result[result['entity_group']!='0'].reset_index(drop=True)
    except:
        dropped = result
    return dropped

def _tag_genes(text):
    _pipe_gene = pipeline("token-classification", model="alvaroalon2/biobert_genetic_ner",aggregation_strategy="first")
    gene_results = _pipe_gene(text)
    df = _drop_unknowns(pd.DataFrame(gene_results))
    df['original_text'] = text
    return df

def _tag_chemicals(text):
    _pipe_chemical = pipeline("token-classification", model="alvaroalon2/biobert_chemical_ner", aggregation_strategy="first")
    chem_results = _pipe_chemical(text)
    df = _drop_unknowns(pd.DataFrame(chem_results))
    df['original_text'] = text
    return df

def _tag_diseases(text):
    _pipe_disease = pipeline("token-classification", model="alvaroalon2/biobert_diseases_ner", aggregation_strategy="first")
    disease_results = _pipe_disease(text)
    df = _drop_unknowns(pd.DataFrame(disease_results))
    df['original_text'] = text
    return df

ModuleNotFoundError: No module named 'inflect'

In [ ]:
# Download link
# https://www.fda.gov/about-fda/oncology-center-excellence/pediatric-oncology-drug-approvals#:~:text=Downloadable%20file%20for%3A-,Pediatric%20Approvals%20Additional%20Information,-Search%3A

In [ ]:
df = pd.read_excel('pediatric_approvals_additional_information_june_01_2025.xlsx')

In [ ]:
df.head()

In [ ]:
df = df.rename(columns={'INDICATION 4       ':'INDICATIONS'})
results = batch(df['INDICATIONS'])

In [ ]:
results['FDA Brand Label'] = None
for idx,row in results.iterrows():
    tdf = df[df['INDICATIONS']==row['original_text']].reset_index(drop=True)
    results.at[idx, 'FDA Brand Label'] = tdf['DRUGS APPROVED FOR PEDIATRIC CANCERS [BRAND NAME] 1'][0]

results



In [ ]:
tdf = results[results['concept_match_type']!=0].reset_index(drop=True)
tdf[tdf['entity_group']=='DISEASE']['FDA Brand Label'].nunique()

In [ ]:
# import inflect

# inflector = inflect.engine()

# def singularize(word):
#     return inflector.singular_noun(word) or word

# results['inflect_test'] = results['word'].apply(singularize)
# results[['word','inflect_test']][0:50]

In [ ]:
tdf

In [ ]:
condensed_results = tdf.groupby('original_text').apply(
    lambda group: pd.Series({
        'GENETIC_LABELS': ' | '.join(group.loc[group['entity_group'] == 'GENETIC', 'concept_label'].unique()),
        'GENETIC_IDS': ' | '.join(group.loc[group['entity_group'] == 'GENETIC', 'concept_id'].unique()),
        'DISEASE_LABELS': ' | '.join(group.loc[group['entity_group'] == 'GENETIC', 'concept_label'].unique()),
        'DISEASE_IDS': ' | '.join(group.loc[group['entity_group'] == 'DISEASE', 'concept_id'].unique())
    })
).reset_index()
condensed_results

In [ ]:
merged_df = pd.merge(
    df,
    condensed_results,
    left_on='INDICATIONS',
    right_on='original_text',
    how='left'
)
merged_df